# 03 · FEM dầm Euler: lắp ghép bằng vòng lặp đơn giản

Notebook này **tự chứa mã** và chạy được trên Colab bằng CPU. Ta dùng đúng bài toán dầm tựa đơn $L=EI=q=1$ với **hai phần tử Hermite**. Mục tiêu là thấy trực tiếp $K^e,f^e\rightarrow K,F\rightarrow$ áp điều kiện biên $\rightarrow$ nghiệm. Chỉ cần biết mảng NumPy, vòng `for` và `np.linalg.solve`.

**Quy ước bậc tự do:** tại mỗi nút có chuyển vị $w_i$ và góc xoay $\theta_i$. Thứ tự toàn cục là $[w_0,\theta_0,w_1,\theta_1,w_2,\theta_2]$.

In [ ]:
import numpy as np
print("NumPy", np.__version__)

## 1. Ôn nhanh: giải một hệ tuyến tính

`@` là nhân ma trận với véc-tơ; `np.linalg.solve(K,F)` giải $Ku=F$. Không nên tự viết $K^{-1}F$.

In [ ]:
K_small = np.array([[2.0, -1.0], [-1.0, 2.0]])
F_small = np.array([1.0, 0.0])
u_small = np.linalg.solve(K_small, F_small)
print("nghiệm:", u_small)
print("kiểm tra K @ u:", K_small @ u_small)

## 2. Một phần tử dầm có bốn bậc tự do

Phần tử nối hai nút, theo thứ tự $[w_i,\theta_i,w_{i+1},\theta_{i+1}]$. Công thức $K^e$ và $f^e$ được dẫn ở [trang FEM dầm](https://nhannguyenminh.github.io/intro2pinns/beam-fem-foundations.html#he-toan-cuc). Ta **nhập trực tiếp** công thức để tập trung vào phép lắp ghép. Với $L=1$ và hai phần tử, $h=1/2$.

In [ ]:
L = 1.0
EI = 1.0
q = 1.0
n = 2                 # số phần tử
h = L / n

Ke = EI / h**3 * np.array([
    [12,   6*h, -12,   6*h],
    [6*h, 4*h*h, -6*h, 2*h*h],
    [-12, -6*h,  12,  -6*h],
    [6*h, 2*h*h, -6*h, 4*h*h]
])
Fe = q*h/12 * np.array([6, h, 6, -h])
print("Ke =\n", Ke)
print("Fe =", Fe)

## 3. Lắp ghép hai phần tử

Toàn dầm có ba nút, tức sáu bậc tự do. Phần tử 0 dùng các chỉ số `[0,1,2,3]`; phần tử 1 dùng `[2,3,4,5]`. Hai phần tử **cùng đóng góp** vào hàng và cột của $w_1,\theta_1$. Vòng lặp dưới đây chỉ làm phép cộng $K[I_a,I_b]\mathrel{+}=K^e[a,b]$ và $F[I_a]\mathrel{+}=f^e[a]$.

In [ ]:
K = np.zeros((6, 6))
F = np.zeros(6)

for e in range(n):
    dofs = [2*e, 2*e+1, 2*e+2, 2*e+3]
    print("phần tử", e, "→ chỉ số toàn cục", dofs)
    for a in range(4):
        F[dofs[a]] += Fe[a]
        for b in range(4):
            K[dofs[a], dofs[b]] += Ke[a, b]

print("K toàn cục =\n", K)
print("F toàn cục =", F)
print("Kiểm tra ô nút chung K[2,2] =", K[2, 2])

**Dừng và giải thích:** vì sao `K[2,2]` bằng tổng hai số 96? Vì sao `F[2]` bằng $1/4+1/4=1/2$? Hãy chỉ đúng hai lần cộng trong vòng `for`.

## 4. Áp điều kiện biên rồi giải

Dầm tựa đơn có $w_0=w_2=0$. Hai góc xoay ở gối **không** bị khóa. Chỉ số đã biết là `[0,4]`; chỉ số cần tìm là `[1,2,3,5]`. Lấy hàng/cột tự do bằng `np.ix_` để giải hệ nhỏ. Vì chuyển vị đã biết bằng 0, vế phải không cần thêm hạng hiệu chỉnh.

In [ ]:
fixed = [0, 4]
free = [1, 2, 3, 5]
d = np.zeros(6)
K_free = K[np.ix_(free, free)]
F_free = F[free]
d[free] = np.linalg.solve(K_free, F_free)

print("[w0, theta0, w1, theta1, w2, theta2] =")
print(d)
print("chuyển vị giữa dầm =", d[2])
print("nghiệm đúng tại giữa =", 5/384)
assert abs(d[2] - 5/384) < 1e-12

## 5. Kiểm tra cân bằng và phản lực

Ở các hàng tự do, $Kd-F$ gần bằng 0. Hai hàng có chuyển vị đã cho cho phản lực gối. Đây là kiểm tra số quan trọng hơn chỉ nhìn vào một giá trị chuyển vị.

In [ ]:
residual = K @ d - F
print("residual ở hàng tự do:", residual[free])
print("phản lực ở hai gối:", residual[fixed])
print("tổng phản lực + tổng tải:", residual[fixed].sum() + q*L)
assert np.max(np.abs(residual[free])) < 1e-10

## 6. Bài tập nhỏ

1. Đổi `q=1.0` thành `q=2.0`, chạy lại từ đầu. Dự đoán chuyển vị giữa dầm tăng bao nhiêu lần, rồi kiểm tra.
2. Tự chỉ ra trong code ba đoạn tương ứng với **ma trận phần tử**, **lắp ghép**, **áp điều kiện biên**.
3. Kiểm tra `K[2,2]` và `F[2]` bằng tay. Bản hai phần tử trùng nghiệm đúng **tại nút giữa**; điều đó không có nghĩa đường võng trong từng phần tử chính xác mọi nơi.

**Nâng cao khi đã vững:** đổi sang bốn phần tử; lúc đó cần sửa kích thước `K,F,d` và danh sách chỉ số gối/tự do.

**Nộp:** notebook chạy lại từ đầu, ảnh hoặc bảng `K,F,d`, và một đoạn ngắn giải thích tại sao phải cộng đóng góp của hai phần tử ở nút chung. Muốn dựng cả đường võng và nghiên cứu hội tụ, xem [notebook tích hợp 05](05-beam-fem-to-pinn.ipynb) sau khi nắm chắc bài này.